In [ ]:
import wandb

sweep_config = {
    'method': 'bayes', # 'grid', 'random', o 'bayes'
    'metric': {
        'name': 'eval/f1_positive', # Queremos optimizar el F1 de la clase IA
        'goal': 'maximize'
    },
    'parameters': {
        'learning_rate': {
            'distribution': 'log_uniform_values',
            'min': 1e-6,
            'max': 1e-4
        },
        'batch_size': {
            'values': [8, 16, 32]
        },
        'grad_acc': {
            'values': [2, 4, 8] # Para ajustar el Effective Batch Size
        },
        'epochs': {
            'value': 5
        },
        'weight_decay': {
            'values': [0.0, 0.01, 0.1]
        }
    }
}

sweep_id = wandb.sweep(sweep_config, project="TFG-T5-Detector")

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # En T5, los logits suelen ser una tupla, tomamos la primera parte
    predictions = np.argmax(logits[0], axis=-1)

    # Aquí calculamos métricas específicas
    # labels: 0=IA (positive), 1=Humano (negative)
    precision, recall, f1, _ = precision_recall_fscore_support(labels.flatten(), predictions.flatten(), average='binary', pos_label=tokenizer.encode("positive")[0])

    return {
        'accuracy': accuracy_score(labels.flatten(), predictions.flatten()),
        'f1_positive': f1,
        'precision_positive': precision,
        'recall_positive': recall
    }

In [ ]:
def train_sweep():
    # Inicializa una nueva ejecución de wandb
    with wandb.init() as run:
        config = wandb.config

        # 1. Preparar modelo y tokenizer (basado en tu código)
        # 2. Configurar TrainingArguments usando config.learning_rate, etc.
        training_args = TrainingArguments(
            output_dir="./temp_results",
            learning_rate=config.learning_rate,
            per_device_train_batch_size=config.batch_size,
            gradient_accumulation_steps=config.grad_acc,
            num_train_epochs=config.epochs,
            weight_decay=config.weight_decay,
            eval_strategy="epoch",
            report_to="wandb", # Esto es vital para que el Sweep vea los resultados
            fp16=True
        )

        # 3. Tu lógica de Trainer...
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_dataset["train"],
            eval_dataset=tokenized_dataset["validation"],
            compute_metrics=compute_metrics # Debes definir esta función para calcular F1
        )

        trainer.train()

In [ ]:
# count=10 significa que probará 10 combinaciones diferentes de hiperparámetros
wandb.agent(sweep_id, function=train_sweep, count=3)

In [ ]:
# 1. Configuración final (Sustituir con los mejores valores del Sweep)
best_config = {
    "lr": 4.5e-5,
    "batch_size": 8,
    "grad_acc": 8,
    "weight_decay": 0.01
}

# 2. Entrenamiento de consolidación
final_args = TrainingArguments(
    output_dir="./model_final_tfg",
    num_train_epochs=5,
    learning_rate=best_config["lr"],
    per_device_train_batch_size=best_config["batch_size"],
    gradient_accumulation_steps=best_config["grad_acc"],
    weight_decay=best_config["weight_decay"],
    fp16=True,
    report_to="none" # Ya no necesitamos trackear este run final en el sweep
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

final_trainer = Trainer(
    model=model,
    args=final_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator
)

final_trainer.train()

# 3. EVALUACIÓN FINAL SOBRE EL CONJUNTO DE TEST
print("\n--- EVALUACIÓN FINAL EN CONJUNTO DE TEST (UNSEEN DATA) ---")
evaluate_model(model, tokenizer, tokenized_dataset["test"])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(y_true, y_pred):
    # Calcular la matriz
    # Aseguramos el orden ['positive', 'negative'] para que coincida con IA y Humano
    labels = ['positive', 'negative']
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    # Configurar la visualización
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['IA (Synthetic)', 'Humano (Real)'],
                yticklabels=['IA (Synthetic)', 'Humano (Real)'])

    plt.title('Matriz de Confusión: T5 Sentinel - Titulares', fontsize=14)
    plt.xlabel('Predicción del Modelo', fontsize=12)
    plt.ylabel('Etiqueta Real', fontsize=12)

    # Guardar para la memoria del TFG
    plt.savefig('matriz_confusion_t5.png', dpi=300, bbox_inches='tight')
    plt.show()

# Para usarlo, simplemente añade esta línea al final de tu función evaluate_model:
# plot_confusion_matrix(true_labels, mapped_predictions)